In [4]:
from pathlib import Path
import json
import difflib

import httpx
from transformers import AutoTokenizer

from mission_control.inference.requests import ModelMessage

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


# Setup


In [5]:
from pathlib import Path
import json
import difflib

import httpx
from transformers import AutoTokenizer

from mission_control.inference.requests import ModelMessage


# ------------------------------------------------------------------
# Local Qwen cache
# ------------------------------------------------------------------

HF_HUB = Path(
    "/Users/lucasleow/Library/CloudStorage/"
    "OneDrive-Personal/LucasTechVault/"
    "Mission Control/"
    "mission-control-local-inference/"
    "hf-cache/hub"
)

MODEL_CACHE = HF_HUB / "models--Qwen--Qwen3.5-0.8B"
SNAPSHOTS_DIR = MODEL_CACHE / "snapshots"

snapshots = [
    path
    for path in SNAPSHOTS_DIR.iterdir()
    if path.is_dir()
]

assert snapshots, f"No snapshots found under {SNAPSHOTS_DIR}"

# Use most recently modified cached snapshot.
MODEL_SNAPSHOT = max(
    snapshots,
    key=lambda path: path.stat().st_mtime,
)

CHAT_TEMPLATE_PATH = MODEL_SNAPSHOT / "chat_template.jinja"

print("Model snapshot:")
print(MODEL_SNAPSHOT)

print("\nChat template:")
print(CHAT_TEMPLATE_PATH)

print("\nTemplate exists:")
print(CHAT_TEMPLATE_PATH.exists())

Model snapshot:
/Users/lucasleow/Library/CloudStorage/OneDrive-Personal/LucasTechVault/Mission Control/mission-control-local-inference/hf-cache/hub/models--Qwen--Qwen3.5-0.8B/snapshots/2fc06364715b967f1860aea9cf38778875588b17

Chat template:
/Users/lucasleow/Library/CloudStorage/OneDrive-Personal/LucasTechVault/Mission Control/mission-control-local-inference/hf-cache/hub/models--Qwen--Qwen3.5-0.8B/snapshots/2fc06364715b967f1860aea9cf38778875588b17/chat_template.jinja

Template exists:
True


## Load Tokenizer Locally


In [6]:
tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_SNAPSHOT),
    local_files_only = True,
)

print("Tokenizer class: ", tokenizer.__class__.__name__)
print("Vocab size: ", len(tokenizer))
print("Special tokens: ", tokenizer.all_special_tokens)

Tokenizer class:  Qwen2Tokenizer
Vocab size:  248077
Special tokens:  ['<|im_end|>', '<|endoftext|>', '<|audio_start|>', '<|audio_end|>', '<|audio_pad|>', '<|image_pad|>', '<|video_pad|>', '<|vision_start|>', '<|vision_end|>']


## Experiment A - Audit Jinja Template


### A1. Print relevant Jinja Lines

Instead of printing entire 8KB template, search for key structural pieces


In [7]:
template_text = CHAT_TEMPLATE_PATH.read_text()

search_terms = [
    "message.role",
    "im_start", "im_end",
    "add_generation_prompt",
    "enable_thinking",
    "<think>", "</think>"
]

print("=" * 80)
print("EXPERIMENT A - RELEVANT JINJA TEMPLATE LINES")
print("=" * 80)

for line_num, line in enumerate(
    template_text.splitlines(),
    start = 1
):
    if any(term in line for term in search_terms):
        print(f"{line_num:03d}: {line}")

EXPERIMENT A - RELEVANT JINJA TEMPLATE LINES
046:     {{- '<|im_start|>system\n' }}
060:     {{- '<|im_end|>\n' }}
064:         {{- '<|im_start|>system\n' + content + '<|im_end|>\n' }}
070:     {%- if ns.multi_step_tool and message.role == "user" %}
083:     {%- if message.role == "system" %}
087:     {%- elif message.role == "user" %}
088:         {{- '<|im_start|>' + message.role + '\n' + content + '<|im_end|>' + '\n' }}
089:     {%- elif message.role == "assistant" %}
094:             {%- if '</think>' in content %}
095:                 {%- set reasoning_content = content.split('</think>')[0].rstrip('\n').split('<think>')[-1].lstrip('\n') %}
096:                 {%- set content = content.split('</think>')[-1].lstrip('\n') %}
101:             {{- '<|im_start|>' + message.role + '\n<think>\n' + reasoning_content + '\n</think>\n\n' + content }}
103:             {{- '<|im_start|>' + message.role + '\n' + content }}
130:         {{- '<|im_end|>\n' }}
131:     {%- elif message.role == "to

## Experiment B - Render Raw String

Pass ModelMessage class through tokenizer

```
Mission Control abstraction
        ↓
Qwen representation
```


### B1. Construct the Messages


In [10]:
model_messages = [
    ModelMessage(
        role="system",
        content=(
            "You are the reasoning engine for Mission Control."
        ),
    ),
    ModelMessage(
        role="user",
        content=(
            "Explain the purpose of ModelGateway "
            "in two sentences."
        ),
    ),
]

messages = [
    message.model_dump() # convert python obj to python dict by Pydantic
    for message in model_messages
]

messages

[{'role': 'system',
  'content': 'You are the reasoning engine for Mission Control.'},
 {'role': 'user',
  'content': 'Explain the purpose of ModelGateway in two sentences.'}]

### B2. Render, but NOT tokenize


In [12]:
rendered_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print("=" * 40)
print("EXPERIMENT B - RAW RENDERED PROMPT")
print("=" * 40)

print(rendered_prompt)


EXPERIMENT B - RAW RENDERED PROMPT
<|im_start|>system
You are the reasoning engine for Mission Control.<|im_end|>
<|im_start|>user
Explain the purpose of ModelGateway in two sentences.<|im_end|>
<|im_start|>assistant
<think>

</think>




In [13]:
print(repr(rendered_prompt))

'<|im_start|>system\nYou are the reasoning engine for Mission Control.<|im_end|>\n<|im_start|>user\nExplain the purpose of ModelGateway in two sentences.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'


## Experiment C - Token Audit

Keep everything similar but set Tokenize = True


In [14]:
input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=False,
)

print("=" * 40)
print("EXPERIMENT C — TOKEN AUDIT")
print("=" * 40)

print("Token count:")
print(len(input_ids))

print("\nToken IDs:")
print(input_ids)

EXPERIMENT C — TOKEN AUDIT
Token count:
2

Token IDs:
{'input_ids': [248045, 8678, 198, 2523, 513, 279, 31626, 4560, 364, 22496, 7533, 13, 248046, 198, 248045, 846, 198, 814, 20139, 279, 7193, 314, 4744, 39365, 303, 1330, 22157, 13, 248046, 198, 248045, 74455, 198, 248068, 271, 248069, 271], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [18]:
print("Special token mapping:")
print(
    json.dumps(
        tokenizer.special_tokens_map,
        indent=2,
        default=str,
    )
)

Special token mapping:
{
  "eos_token": "<|im_end|>",
  "pad_token": "<|endoftext|>",
  "audio_bos_token": "<|audio_start|>",
  "audio_eos_token": "<|audio_end|>",
  "audio_token": "<|audio_pad|>",
  "image_token": "<|image_pad|>",
  "video_token": "<|video_pad|>",
  "vision_bos_token": "<|vision_start|>",
  "vision_eos_token": "<|vision_end|>"
}


## Experiment D - Break Contract & Toggle Thinking


### Setup - Find local vLLM

Make sure to start local vLLM first.

```
source ~/.venv-vllm-metal/bin/activate

export HF_HOME="$HOME/mission-control-local-inference/hf-cache"

VLLM_METAL_MEMORY_FRACTION=0.70 \
vllm serve Qwen/Qwen3.5-0.8B \
    --host 127.0.0.1 \
    --port 8000 \
    --max-model-len 4096

```


In [19]:
VLLM_BASE_URL = "http://127.0.0.1:8000/v1"

response = httpx.get(
    f"{VLLM_BASE_URL}/models",
    timeout=10.0,
)

response.raise_for_status()

available_models = response.json()["data"]

for model in available_models:
    print(model["id"])

SERVED_MODEL = available_models[0]["id"]

print("\nUsing:")
print(SERVED_MODEL)

Qwen/Qwen3.5-0.8B

Using:
Qwen/Qwen3.5-0.8B


### Experiment D1 - Naive Prompt vs Correct Template


In [20]:
correct_template_messages = [
    {
        "role": "system",
        "content": (
            "You are Mission Control. "
            "Always prefix your final response with 'MC:'."
        ),
    },
    {
        "role": "user",
        "content": (
            "Explain what an inference server does "
            "in exactly one sentence."
        ),
    },
]

correct_prompt = tokenizer.apply_chat_template(
    correct_template_messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

In [22]:
naive_prompt = """System:
You are Mission Control.
Always prefix your final response with 'MC:'.

User:
Explain what an inference server does in exactly one sentence.

Assistant:
"""

# Helper
def raw_completion(prompt: str) -> dict:
    response = httpx.post(
        f"{VLLM_BASE_URL}/completions",
        json={
            "model": SERVED_MODEL,
            "prompt": prompt,
            "temperature": 0.0,
            "max_tokens": 128,
        },
        timeout=60.0,
    )

    response.raise_for_status()

    return response.json()

In [23]:
correct_result = raw_completion(correct_prompt)
naive_result = raw_completion(naive_prompt)

In [24]:
# Compare Results
correct_text = correct_result["choices"][0]["text"]
naive_text = naive_result["choices"][0]["text"]

print("=" * 80)
print("CORRECT QWEN TEMPLATE")
print("=" * 80)
print(correct_text)

print("\n" + "=" * 80)
print("NAIVE HAND-WRITTEN FORMAT")
print("=" * 80)
print(naive_text)

CORRECT QWEN TEMPLATE
MC: An inference server processes large datasets to generate predictions or answers for users without storing them permanently, significantly reducing storage costs and latency.

NAIVE HAND-WRITTEN FORMAT
It processes data from multiple sources to generate new insights and answers questions.
MC:
<think>

</think>

MC:
It processes data from multiple sources to generate new insights and answers questions.


In [25]:
def inspect_output(name: str, text: str) -> None:
    sentences = [
        s.strip()
        for s in text.replace("!", ".").replace("?", ".").split(".")
        if s.strip()
    ]

    print(f"\n{name}")
    print("-" * 40)
    print("Starts with MC:       ", text.strip().startswith("MC:"))
    print("Approx sentence count:", len(sentences))
    print("Output                :", repr(text))


inspect_output(
    "Correct template",
    correct_text,
)

inspect_output(
    "Naive template",
    naive_text,
)


Correct template
----------------------------------------
Starts with MC:        True
Approx sentence count: 1
Output                : 'MC: An inference server processes large datasets to generate predictions or answers for users without storing them permanently, significantly reducing storage costs and latency.'

Naive template
----------------------------------------
Starts with MC:        False
Approx sentence count: 2
Output                : 'It processes data from multiple sources to generate new insights and answers questions.\nMC:\n<think>\n\n</think>\n\nMC:\nIt processes data from multiple sources to generate new insights and answers questions.'


### Experiment D2 - Thinking ON vs Thinking OFF

Smaller Qwen models can be prone to thinking loops than larger Qwen variants.


In [26]:
thinking_messages = [
    {
        "role": "user",
        "content": (
            'Reverse the string "MISSION". '
            "Return only the reversed string."
        ),
    },
]


thinking_off_prompt = tokenizer.apply_chat_template(
    thinking_messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)


thinking_on_prompt = tokenizer.apply_chat_template(
    thinking_messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

In [32]:
print("=" * 40)
print("THINKING OFF — RENDERED INPUT")
print("=" * 40)

print(thinking_off_prompt)

print("\n" + "=" * 40)
print("THINKING ON — RENDERED INPUT")
print("=" * 40)

print(thinking_on_prompt)

THINKING OFF — RENDERED INPUT
<|im_start|>user
Reverse the string "MISSION". Return only the reversed string.<|im_end|>
<|im_start|>assistant
<think>

</think>



THINKING ON — RENDERED INPUT
<|im_start|>user
Reverse the string "MISSION". Return only the reversed string.<|im_end|>
<|im_start|>assistant
<think>



In [28]:
diff = difflib.unified_diff(
    thinking_off_prompt.splitlines(),
    thinking_on_prompt.splitlines(),
    fromfile="thinking=False",
    tofile="thinking=True",
    lineterm="",
)

print("\n".join(diff))

--- thinking=False
+++ thinking=True
@@ -2,6 +2,3 @@
 Reverse the string "MISSION". Return only the reversed string.<|im_end|>
 <|im_start|>assistant
 <think>
-
-</think>
-


In [29]:
off_result = raw_completion(
    thinking_off_prompt
)

on_result = raw_completion(
    thinking_on_prompt
)

In [33]:
print("=" * 40)
print("THINKING OFF — MODEL OUTPUT")
print("=" * 40)

print(
    off_result["choices"][0]["text"]
)

print("\nUsage:")
print(off_result.get("usage"))


print("\n" + "=" * 40)
print("THINKING ON — MODEL OUTPUT")
print("=" * 40)

print(
    on_result["choices"][0]["text"]
)

print("\nUsage:")
print(on_result.get("usage"))

THINKING OFF — MODEL OUTPUT
MISSION

Usage:
{'prompt_tokens': 24, 'total_tokens': 26, 'completion_tokens': 2, 'prompt_tokens_details': None}

THINKING ON — MODEL OUTPUT
Thinking Process:

1.  **Analyze the Request:**
    *   Input string: "MISSION"
    *   Task: Reverse the string.
    *   Constraint: Return *only* the reversed string.

2.  **Perform the Reversal:**
    *   Original: M I S I O N
    *   Last character: N
    *   Second to last: I
    *   Third to last: O
    *   Fourth to last: S
    *   Fifth to last: I
    *   First to last: M
    *   Wait

Usage:
{'prompt_tokens': 22, 'total_tokens': 150, 'completion_tokens': 128, 'prompt_tokens_details': None}


In [31]:
d2_results = {
    "thinking_off": {
        "output": off_result["choices"][0]["text"],
        "finish_reason": off_result["choices"][0]["finish_reason"],
        "usage": off_result.get("usage"),
    },
    "thinking_on": {
        "output": on_result["choices"][0]["text"],
        "finish_reason": on_result["choices"][0]["finish_reason"],
        "usage": on_result.get("usage"),
    },
}

d2_results

{'thinking_off': {'output': 'MISSION',
  'finish_reason': 'stop',
  'usage': {'prompt_tokens': 24,
   'total_tokens': 26,
   'completion_tokens': 2,
   'prompt_tokens_details': None}},
 'thinking_on': {'output': 'Thinking Process:\n\n1.  **Analyze the Request:**\n    *   Input string: "MISSION"\n    *   Task: Reverse the string.\n    *   Constraint: Return *only* the reversed string.\n\n2.  **Perform the Reversal:**\n    *   Original: M I S I O N\n    *   Last character: N\n    *   Second to last: I\n    *   Third to last: O\n    *   Fourth to last: S\n    *   Fifth to last: I\n    *   First to last: M\n    *   Wait',
  'finish_reason': 'length',
  'usage': {'prompt_tokens': 22,
   'total_tokens': 150,
   'completion_tokens': 128,
   'prompt_tokens_details': None}}}